In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "brauer2015apes")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Braeuer_2015_2008braeuer-toolproduction1-120sec-alldata.sav")
complete_path_2 = os.path.join(original_data_pathway, "Braeuer_2015_2010braeuer-toolproduction2-50sec-alldata.sav")
complete_path_3 = os.path.join(original_data_pathway, "Braeuer_2015_2010braeuer-toolproduction3-50sec-alldata.sav")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
df2 = pd.read_spss(complete_path_2, usecols=None, convert_categoricals=True)
df3 = pd.read_spss(complete_path_3, usecols=None, convert_categoricals=True)
experiment_import = [[df1, '120_sec','1'],
                    [df2, '50_sec_1', '1'],
                    [df3, '50_sec_2','2']]
for x, y, k in experiment_import:
    x['production']=y
    x['group_original'] = k


In [3]:
data_frames=[df1, df2, df3]

for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x=x.rename(columns={"subject": "ape"})
    x=x.rename(columns={"subject_string": "ape"})
    x['study_id']="brauer2015apes"
    data_frames[index]=x
new_df=data_frames[0]

In [4]:
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

fulldf['ape'] = fulldf['ape'].str.rstrip()

fulldf[['ape','present']] = fulldf['ape'].str.split('+', expand=True)
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)
    fulldf['present'].replace(x, y, inplace=True)


In [5]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')
# fulldf.columns

In [6]:
fulldf=fulldf[fulldf['ape'].notna()]
fulldf['experiment'] = 1
fulldf.rename(columns={"ape": "participant",
                       "cond":"condition",
                       "group_original":"group"}, inplace=True)

complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
fulldf= fulldf.merge(subject_list,left_on='participant', right_on='name', how='left')
fulldf.rename(columns={"age": "age_in_years"}, inplace=True)

In [7]:

brauer2015apes_standardized=fulldf[['study_id', 'experiment', 
                                    'participant','age_in_years', 'sex', 'species',  'present',
       'condition', 'group','production','tools_prepared', 'grapes_eaten', 'success'
       ]]

In [8]:
comp_out_path_stand = os.path.join(out_pathway, 'brauer2015apes_standardized.csv')
brauer2015apes_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

In [9]:
names = brauer2015apes_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'brauer2015apes_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)